In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
TASK = Path.cwd().parent.parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output/exploratory'
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
MODELS = CONFIG['models']
PRIMARY = CONFIG['primary_plot_models']
LABELS = {'arw4':'ARW4', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'AR(1)-H', 'ar6_6':'AR(6,6)', 'se_hurdle_s':'SE-H'}
SECONDARY = OUTPUT/'secondary'
SECONDARY.mkdir(exist_ok=True)
VIEWS = [(OUTPUT,PRIMARY), (SECONDARY,MODELS)]
palette = sns.color_palette('colorblind',6)
COLORS = {source:palette[i] for source,i in zip(LABELS,[3,0,1,4,5,2])}
MARKERS = dict(zip(LABELS,['P','o','s','^','D','X']))
DASHES = dict(zip(LABELS,[(3,1,1,1),'',(4,2),(2,1),(6,2,1,2),(8,2)]))

In [3]:
distances = pd.read_csv(INPUT/'annual_wasserstein.csv')
generated_from = distances.groupby('source').generated_from_age.first()

In [4]:
sns.set_theme(style='ticks',context='paper',rc={'pdf.fonttype':42})
for dest,selected in VIEWS:
    fig,ax = plt.subplots(figsize=(7.2,4.1))
    if 'ar6_6' in selected:
        ax.axvspan(0, generated_from['ar6_6']-.5, color='.9', zorder=0, label='AR(6,6) initialized')
    ax.axhline(0,color='.3',lw=.8,ls=':',label='DBLP reference')
    sns.lineplot(data=distances.loc[distances.source.isin(selected)],x='age',y='wasserstein1',
                 hue='source',style='source',hue_order=selected,style_order=selected,
                 palette=COLORS,markers=MARKERS,dashes=DASHES,markersize=4,
                 estimator=None,errorbar=None,ax=ax)
    handles,names = ax.get_legend_handles_labels()
    ax.legend(handles,[LABELS.get(name,name) for name in names],frameon=False,ncol=2,title=None)
    ax.set(xlabel='Career year',ylabel='Wasserstein-1 (papers)',xlim=(0,Y),
           title='Annual productivity distance to DBLP')
    ax.set_xticks(range(0,Y+1,2))
    ax.set_yscale('symlog',linthresh=1)
    ax.set_ylim(bottom=-.06)
    sns.despine(ax=ax)
    for ext in ['pdf','png']:
        fig.savefig(dest/f'annual_wasserstein.{ext}',bbox_inches='tight',dpi=180,facecolor='white')
    plt.close(fig)